In [25]:
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense ,Dropout
import kerastuner as kt


In [2]:
df = pd.read_csv('diabetes.csv')

In [3]:
X = df.drop('Outcome',axis=1)
Y = df['Outcome']

In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scale = scaler.fit_transform(X)

In [5]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(X_scale,Y,test_size=0.2,random_state=42)

In [6]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(72,activation='relu',input_dim=8))
    # in this loop there are 2 loop 
    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):
        model.add(Dense(72,activation='relu'))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [7]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=3,
    directory = 'layers',
    project_name='testing'
    
)

Reloading Tuner from layers/testing/tuner0.json


In [8]:
tuner.search(X_train,Y_train,epochs=5,validation_data=(X_test,Y_test))

In [9]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7}

In [10]:
tuner.results_summary()

Results summary
Results in layers/testing
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
num_layers: 7
Score: 0.7857142686843872

Trial 2 summary
Hyperparameters:
num_layers: 4
Score: 0.7727272510528564

Trial 0 summary
Hyperparameters:
num_layers: 2
Score: 0.7467532753944397


In [11]:
model = tuner.get_best_models(num_models=1)[0]

/Users/All file hear/complete ai /Deep learning/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Users/All file hear/complete ai /Deep learning/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 20 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [12]:
model.fit(X_train,Y_train,epochs=100,initial_epoch=5,validation_data=(X_test,Y_test))

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7769 - loss: 0.4661 - val_accuracy: 0.7468 - val_loss: 0.5523
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8094 - loss: 0.4194 - val_accuracy: 0.7792 - val_loss: 0.6295
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7964 - loss: 0.4182 - val_accuracy: 0.7338 - val_loss: 0.5286
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8274 - loss: 0.4012 - val_accuracy: 0.7532 - val_loss: 0.6156
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8127 - loss: 0.3961 - val_accuracy: 0.7532 - val_loss: 0.5802
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8111 - loss: 0.3880 - val_accuracy: 0.7078 - val_loss: 0.5872
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8339 - loss: 0.3689 - val_accuracy: 0.7273 - val_loss: 0.6802
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8322 - loss: 0.3744 - val_accuracy: 0.7468

In [28]:
def build_model2(hp):
    model = Sequential()
    counter = 0



    for i in range (hp.Int('num_layers',min_value=1,max_value=10)):
        if counter == 0:
            model.add(Dense(hp.Int('units'+str(i),min_value= 8,max_value =128,step=8),
                            activation=hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid']),
                            input_dim=8))
            model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        else:
            model.add(Dense(hp.Int('units'+str(i),min_value= 8,max_value =128,step=8),
                            activation=hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid'])))
            model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        counter+=1


    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer=hp.Choice('optimizer',values=['rmsprop','adam','sgd','nadam','adadelta']
                                      ),loss='binary_crossentropy',
                                      metrics=['accuracy'])
    return model
    





In [30]:
tuner2 = kt.RandomSearch(build_model2,
                        objective='val_accuracy',
                        max_trials=3,
                        directory='combine3',
                        project_name='final3')

In [31]:
tuner2.search(X_train ,Y_train,epochs=5,validation_data=(X_test,Y_test))

Trial 3 Complete [00h 00m 02s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.6428571343421936
Total elapsed time: 00h 00m 05s


In [32]:
tuner.results_summary()

Results summary
Results in layers/testing
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
num_layers: 7
Score: 0.7857142686843872

Trial 2 summary
Hyperparameters:
num_layers: 4
Score: 0.7727272510528564

Trial 0 summary
Hyperparameters:
num_layers: 2
Score: 0.7467532753944397


In [21]:
tuner2.get_best_hyperparameters()[0].values

{'num_layers': 9,
 'units0': 8,
 'activation0': 'tanh',
 'optimizer': 'nadam',
 'units1': 56,
 'activation1': 'sigmoid',
 'units2': 56,
 'activation2': 'tanh',
 'units3': 64,
 'activation3': 'tanh',
 'units4': 40,
 'activation4': 'tanh',
 'units5': 96,
 'activation5': 'tanh',
 'units6': 8,
 'activation6': 'relu',
 'units7': 128,
 'activation7': 'tanh',
 'units8': 88,
 'activation8': 'tanh',
 'units9': 24,
 'activation9': 'relu'}

In [34]:
model2 = tuner2.get_best_models(num_models=1)[0]

In [36]:
from tensorflow.keras.callbacks import EarlyStopping

In [45]:
ES = EarlyStopping(
    monitor='val_accuracy',
    min_delta=0,
    mode='auto',
    restore_best_weights=True,
    baseline=None,
    verbose=1,
    patience=50
)

In [46]:
model2.fit(X_train,Y_train,epochs=200,initial_epoch=5,validation_data=(X_test,Y_test),callbacks=ES)

Epoch 6/200


20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8111 - loss: 0.4094 - val_accuracy: 0.7338 - val_loss: 0.5726
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8225 - loss: 0.4105 - val_accuracy: 0.7338 - val_loss: 0.5694
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8339 - loss: 0.3982 - val_accuracy: 0.7273 - val_loss: 0.5760
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8290 - loss: 0.3980 - val_accuracy: 0.7143 - val_loss: 0.5858
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8388 - loss: 0.4080 - val_accuracy: 0.7273 - val_loss: 0.5726
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7997 - loss: 0.4323 - val_accuracy: 0.7013 - val_loss: 0.5774
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8176 - loss: 0.4124 - val_accuracy: 0.7078 - val_loss: 0.5847
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8192 - loss: 0.3925 - val_accuracy: 0.7143 - val_loss:

In [65]:
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)
y_pred

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


array([[0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [1],
       [1],
       [1],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [1],
       [1],
       [0],
       [1],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
    

In [67]:
from sklearn.metrics import accuracy_score


In [68]:
Y_test

668    0
324    0
624    0
690    0
473    0
      ..
355    1
534    0
344    0
296    1
462    0
Name: Outcome, Length: 154, dtype: int64

In [69]:
accuracy_score(Y_test,y_pred)

0.7597402597402597